# Quantum State Preparation
Implementation of the quantum state preparation problem in Qiskit. Given a $2^{n}$ dimensional vector of complex amplitudes, with $\| \psi \|_2=1$ , the code constructs a $U$ such that $$U\ket{0}_{n} = \sum_{x=0}^{2^n-1}\psi_{x}\ket{x}_n$$This problem can be broken down into two aprts: amplitude preparation and phase preparation. $$U\ket{0}_{n} = \sum_{x=0}^{2^n-1}\psi_{x}\ket{x}_n = \sum_{x=0}^{2^n-1}|\psi_{x}|e^{-i\phi_x}\ket{x}_n = D_{\bar{\phi}}\sum_{x=0}^{2^n-1}|\psi_{x}|\ket{x}_n$$, where $ D_{\bar{\phi}} = diag(\phi_1, \phi_2, .. \phi_{2^n-1}) $ is a diagonal unitary, and $|\psi_{x}|> 0$.

So we first prepare the amplitudes and then apply a diagonal unitary via multiplexed $R_z$ rotations to obtain the full state. 

## Amplitude preparation:
The idea behind amplitude preparation can be understood by examining a 3 qubit state. Let us consider the general state, $$\ket{\psi} = a \ket{000} + b \ket{100} + c\ket{010} + d\ket{110} + e\ket{001} + f\ket{101} + g\ket{011} + h\ket{111}.$$ This state can be factorised as follows:
$$
\begin{aligned}
\ket{\psi} = {}& \frac{\sqrt{a^2+b^2+c^2+d^2}}{\sqrt{a^2+b^2+c^2+d^2+e^2+f^2+g^2+h^2}}\,\ket{0} \Bigg[ \\
&\quad \frac{\sqrt{a^2+b^2}}{\sqrt{a^2+b^2+c^2+d^2}}\,\ket{0} \left(\frac{a}{\sqrt{a^2+b^2}}\ket{0} + \frac{b}{\sqrt{a^2+b^2}}\ket{1}\right) \\
&\quad + \frac{\sqrt{c^2+d^2}}{\sqrt{a^2+b^2+c^2+d^2}}\,\ket{1} \left(\frac{c}{\sqrt{c^2+d^2}}\ket{0} + \frac{d}{\sqrt{c^2+d^2}}\ket{1}\right) \Bigg]
\end{aligned}
$$
$$
\begin{aligned}
+\ {}& \frac{\sqrt{e^2+f^2+g^2+h^2}}{\sqrt{a^2+b^2+c^2+d^2+e^2+f^2+g^2+h^2}}\,\ket{1} \Bigg[ \\
&\quad \frac{\sqrt{e^2+f^2}}{\sqrt{e^2+f^2+g^2+h^2}}\,\ket{0} \left(\frac{e}{\sqrt{e^2+f^2}}\ket{0} + \frac{f}{\sqrt{e^2+f^2}}\ket{1}\right) \\
&\quad + \frac{\sqrt{g^2+h^2}}{\sqrt{e^2+f^2+g^2+h^2}}\,\ket{1} \left(\frac{g}{\sqrt{g^2+h^2}}\ket{0} + \frac{h}{\sqrt{g^2+h^2}}\ket{1}\right) \Bigg]
\end{aligned}
$$

Here, we have factorized from the most siginificant bit (leftmost) first and so on and have done so in such a way that the coefficients can be identified as sins and cosines of specific angles. By inspection, we can identify, 

$$
\begin{aligned}
\textbf{i=0:}\quad \theta_{0,0} &= 2\arccos\!\left(\frac{\sqrt{a^2+b^2+c^2+d^2}}{\sqrt{a^2+b^2+c^2+d^2+e^2+f^2+g^2+h^2}}\right) \\[6pt]
\textbf{i=1:}\quad \theta_{1,0} &= 2\arccos\!\left(\frac{\sqrt{a^2+b^2}}{\sqrt{a^2+b^2+c^2+d^2}}\right) \\
\theta_{1,1} &= 2\arccos\!\left(\frac{\sqrt{e^2+f^2}}{\sqrt{e^2+f^2+g^2+h^2}}\right) \\[6pt]
\textbf{i=2:}\quad \theta_{2,0} &= 2\arccos\!\left(\frac{a}{\sqrt{a^2+b^2}}\right), \qquad
\theta_{2,1} = 2\arccos\!\left(\frac{c}{\sqrt{c^2+d^2}}\right) \\
\theta_{2,2} &= 2\arccos\!\left(\frac{e}{\sqrt{e^2+f^2}}\right), \qquad
\theta_{2,3} = 2\arccos\!\left(\frac{g}{\sqrt{g^2+h^2}}\right)
\end{aligned}
$$

The pattern is to break the list of amplitudes in $2^i$ blocks at the $i^{\text{th}}$ iteration, and the angles can be found as the ratio of the norms of the half block to the full block.

## Phase preparation:
...

In [ ]:
#Imports

from qiskit.circuit import QuantumCircuit, QuantumRegister, AncillaRegister, Parameter
from qiskit.quantum_info import Statevector, Operator, state_fidelity
from qiskit.circuit.library.standard_gates import RYGate, RZGate

import numpy as np

In [ ]:
# HELPER FUNCTION 1
# Calculating angles for the Ry and Rz rotations to prepare the amplitudes and phases respectively
# Every i^th step, split the amplitude vector into 2^i blocks, calculate one angle for each block 
# In the m^th block of the i^th iteration, the angle is calculates as 
# theta_amps[i,m] = 2 * arccos(norm of the first half of the block/norm of the full block)
# theta_phases[i,m] = mean of the second half - mean of the first half

# The loops i go from 0 to n-1 for a total of n iterations for n qubits, and m goes from 1 to 2^i.

def find_thetas(amps, phases, n):
    """
    Calculate the angles for the Ry rotations to prepare the amplitudes and the Rz rotations to prepare the phases.
    Args:
        amps (np.ndarray): 2^n dimensional vector of (real, positive) amplitudes.
        phases (np.ndarray): 2^n dimensional vector of phases in radians.
        n (int): Number of qubits.
    Returns:
        thetas_amps (np.ndarray): 2D array of angles for the Ry rotations
        thetas_phases (np.ndarray): 2D array of angles for the Rz rotations
    """
    thetas_amps = np.zeros((n, 2**(n-1))) # store the angles for each block in a 2D array, the first index is the iteration, the second is the block number
    thetas_phases = np.zeros((n, 2**(n-1)))
    for i in range(n):

        blocks_amps = np.split(amps, 2**i) # split the amplitude vector into 2^i blocks
        blocks_phases = np.split(phases, 2**i) # split the phase vector into 2^i blocks

        for m, block in enumerate(blocks_amps):
            first_half = block[:len(block)//2]            
            if np.linalg.norm(block) != 0:
                thetas_amps[i][m] = 2 * np.arccos(np.linalg.norm(first_half)/np.linalg.norm(block))
            else: 
                thetas_amps[i][m] = 0

        for m, block in enumerate(blocks_phases):
            first_half = block[:len(block)//2]
            second_half = block[len(block)//2:]
            thetas_phases[i][m] = np.mean(second_half) - np.mean(first_half)
            
    return thetas_amps, thetas_phases    

In [ ]:
# HELPER FUNCTION 2 
# with theta[i][m] we need to construct a multiplexed R_y rotation(or R_z, using R_y from here throughout, exactly same 
# structure for the R_z rotations to implement phases, which is why we are abstracting this out as a function) 

# On the n-1+i^th qubit (starting at the bottommost/leftmost qubit and working our way up the circuit), controlled by i qubits.
# Note that we have i+1 angles for the i^th R_y rotation, so for i=0, on the bottommost qubit, we have 1 angle, 
# we apply a simple R_y rotation here, 0 control qubits. For i = 1, we have 2 angles, we apply multiplexed Ry on 
# the n-2^th qubit, controlled on the n-1^th qubit, each angle corresponsing to the control qubit being 0 or 1 respectivly. 
# FOr i=2, we have 4 angles, we apply multiplexed Ry on the n-3 qubit, controlled on the n-2 and n-1 qubits, 
# each angle corresponding to the control qubits being 00, 01, 10, 11 respectively.

def multiplexed_rotations_for_state_prep(n, thetas, gate, quantum_register, state_prep_circ):
    """
    Constructs a quantum circuit for state preparation using multiplexed rotations.

    Args:
        n (int): Number of qubits.
        thetas (list): List of angles for the rotations.
        gate (Gate): The rotation gate to be used (e.g., RYGate, RZGate).
        quantum_register (QuantumRegister): The quantum register to apply the rotations on.
        state_prep_circ (QuantumCircuit): The quantum circuit to append the rotations to.
    Returns:
        QuantumCircuit: The quantum circuit with the multiplexed rotations applied.
    """
 
    for i in range(n):

        if i == 0:
            state_prep_circ.append(gate(thetas[0][0]), [quantum_register[n-1]]) # apply the first Ry rotation on the bottommost qubit, with no control qubits
            state_prep_circ.barrier()

        else:
            num_control_qubits = i
            control_qubits = quantum_register[n-1 : n-1-i : -1] 
            control_bitstrings = [f'{m:0{num_control_qubits}b}' for m in range(2**num_control_qubits)] #enumerate control qubit bitstrings 
            
            # The multiplexed Ry requires choosing the rotation angle depending on the control bitstring. We do this 
            # by looping over the 2^i-1 possible (angle, bitstring) pairs at every i^th step, and
            # for the m^th angle corresponding to the m^th control bitstring, we turn the multiplexed Ry as a simpler
            # multicontrolled Ry using an x bitmask on the control qubits.  
            for m in range(2**num_control_qubits):
                zero_qubits = []
                for b, q in zip(control_bitstrings[m], control_qubits):
                    if b=='0':
                        zero_qubits += [q] 
                
                if zero_qubits:
                    #apply x gate in stat_prep_circ if the control bit above is zero. 
                    state_prep_circ.x(zero_qubits)

                #apply the multiplexed Ry rotation on the target qubit, controlled by the control qubits, with the angles calculated above.
                # append takes qubit arguments as a single list, target, then controls
            
                state_prep_circ.append(gate(thetas[i][m]).control(i),control_qubits+[quantum_register[n-1-i]])

                if zero_qubits:
                    #apply x gate in stat_prep_circ to the same qubits as above to flip them back
                    state_prep_circ.x(zero_qubits)

                state_prep_circ.barrier()
                

In [ ]:
# MAIN

# FOR DEMO, set number of qubits, code works for general n
n = 3

# Input an n qubit state as a 2^n dimensional vector of amplitudes, assuming little-endian convention
input_psi = np.random.randn(2**n) #+ 1j* np.random.randn(2**n)

# OR input amplitude vector directly
#input_psi = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16])

if np.linalg.norm(input_psi) != 1:
    input_psi = input_psi/np.linalg.norm(input_psi) # normalize 

# Note that the default qiskit convention is little endian. Which is what we use throughout this project
# So for example, for two qubits, the ordering of amplitudes is 00, 01, 10, 11
# The first qubit, q0, the rightmost one, which is the topmost in a circuit, is the least significant and flips first
target_initial_state = Statevector(input_psi) 

n = int(np.log2(len(input_psi))) # number of qubits 
print(n)

# Split into real amplitudes and phases, these are implemented separately 
amps = np.abs(input_psi) #2^n dimensional vector of (real, positive) amplitudes
phases = np.angle(input_psi) #2^n dimensional vector of phases in radians

thetas_amps, thetas_phases = find_thetas(amps, phases, n)

# Construct the quantum circuit
quantum_register = QuantumRegister(size=n, name="qreg")

state_prep_circ = QuantumCircuit(quantum_register, name="Quantum State Preparation")

# Apply multiplexed rotations for amplitude preparation, returns circuit after amplitude prep
multiplexed_rotations_for_state_prep(n, thetas_amps, RYGate, quantum_register, state_prep_circ)   

# Apply multiplexed rotations for phase preparation, returns final circuit after phase prep
multiplexed_rotations_for_state_prep(n, thetas_phases, RZGate, quantum_register, state_prep_circ)

#Compare the final state of the circuit with the target state
prepared_state = Statevector(state_prep_circ)
target_initial_state = Statevector(input_psi)

# Calculate the fidelity between the prepared state and the target state
# (fidelity ignores the global phase)
fidelity = state_fidelity(prepared_state, target_initial_state) 
print(f"Fidelity between the prepared state and the target state: {fidelity}")

In [ ]:
# Draw the full quantum circuit
state_prep_circ.draw(output='mpl')